# Notebook 6: Confidence Intervals & Effect Sizes

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours  
**Theme:** House Price Prediction  
**Week 3 — Statistics for ML & Python Data Stack**

---

## Learning Objectives
By the end of this notebook you will be able to:
- State the **correct** interpretation of a confidence interval
- Compute a 95% CI manually using the t-distribution formula
- Explain how sample size and variability determine CI width
- Calculate **Cohen's d** as a standardised measure of effect size
- Classify effect sizes as small (0.2), medium (0.5), or large (0.8)
- Recognise the "significant but trivial" paradox and know how to identify it
- Interpret overlapping vs non-overlapping confidence intervals


## 1. Why Does This Matter for Machine Learning?

In Notebook 5 we learned that p-values only answer the question: *"Is there an effect?"* (binary: yes/no, based on a threshold).

But in ML, we need richer answers:
- "By how much does adding this feature improve RMSE?"
- "What range of values is plausible for the true model accuracy?"
- "Is the difference between Model A and Model B large enough to justify switching?"
- "How certain are we about our estimate of the average house price?"

**Confidence intervals** and **effect sizes** answer these questions.

### Real ML Scenarios:

| Scenario | P-value tells you | CI + effect size tells you |
|----------|------------------|---------------------------|
| Model A vs Model B | Is A different from B? | A is better by [0.2%, 1.8%] with d=0.3 |
| Adding "school quality" feature | Does it help? | It reduces RMSE by [$500, $3,200] |
| District price comparison | Do districts differ? | District A costs [$15K, $35K] more |
| Sample size planning | N/A | Need n≥200 to detect a 5% difference |

> **Key insight:** CIs and effect sizes make your findings **actionable**. A p-value alone almost never is.


## 2. The GPS Analogy for Confidence Intervals

Imagine your GPS says: *"You are at this exact point on the map."*

But GPS isn't perfect — it has measurement error. A more honest GPS would say: *"You are **within 50 meters** of this location."*

That 50-meter circle around your estimated position is a **confidence interval** — it's a range of plausible true values, not a single point.

**Translate to house prices:**
- You sample 50 houses from a neighbourhood and compute the mean price: $307,500.
- But $307,500 is just **your sample's mean** — the true population mean is unknown.
- A 95% CI says: *"Based on this sample, the true mean is plausibly somewhere between $295,000 and $320,000."*

### The Critical Point About GPS

If 100 different GPS devices all measured your position and each drew a 50-meter circle:
- About **95** of those circles would contain your true position
- About **5** would miss — the device happened to measure on the unlucky side

That's exactly what a 95% CI means — and it's **about the procedure, not about any single interval.**


## 3. Plain English: What IS a Confidence Interval?

A **95% confidence interval** is a range computed from sample data using a procedure that, **if repeated many times across many different samples**, would contain the true population parameter in approximately 95% of those repetitions.

### The Correct Interpretation

> *"We are 95% confident in the **procedure** that produced this interval. If we repeated this study 100 times, about 95 of the resulting intervals would contain the true mean."*

### The Common Misinterpretation (WRONG)

> ~~"There is a 95% probability that the true mean is inside this specific interval."~~

Why is this wrong? Because once you've computed a specific interval (say [$295K, $320K]), the true mean either IS or IS NOT in that interval — there's no probability involved for a fixed interval. The probability statement applies to the **process** of creating intervals, not to any single interval.

### The Formula

$$\text{CI} = \bar{x} \pm t^*_{\alpha/2, \, df} \times SE$$

Where:
- $\bar{x}$ = sample mean (your best point estimate)
- $t^*_{\alpha/2, df}$ = critical t-value for the desired confidence level and degrees of freedom
- $SE = \dfrac{s}{\sqrt{n}}$ = standard error of the mean
- $s$ = sample standard deviation
- $n$ = sample size
- $df = n - 1$ = degrees of freedom

For large n (n > 30), the t-distribution approaches the normal distribution, and $t^*_{0.025} \approx 1.96$.

### Margin of Error

$$\text{Margin of Error} = t^* \times SE = t^* \times \frac{s}{\sqrt{n}}$$

The CI gets **narrower** (more precise) when:
- $n$ increases (more data)
- $s$ decreases (less variability in the population)
- Confidence level decreases (e.g., 90% CI is narrower than 95% CI)


In [ ]:
# ============================================================
# SETUP: Import all required libraries
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Visual settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# ============================================================
# Define the "true" population we're sampling from
# In reality we never know these — this is our simulation ground truth
# ============================================================
TRUE_MEAN = 300_000      # $300,000 true mean house price in the city
TRUE_STD  = 50_000       # $50,000 true standard deviation of house prices

print("Libraries loaded successfully!")
print()
print("Simulation parameters:")
print(f"  True population mean: ${TRUE_MEAN:,}")
print(f"  True population std:  ${TRUE_STD:,}")
print()
print("(In practice we never know the true mean — that's WHY we need CIs)")

## 4. Computing a 95% CI — Step by Step

Let's walk through every step of computing a 95% CI for mean house price.


In [ ]:
# ============================================================
# STEP-BY-STEP: Compute 95% CI for mean house price from n=50
# ============================================================

np.random.seed(42)
N_SAMPLE = 50    # we sample 50 houses — realistic for a field survey

# Step 1: Draw a random sample from the city's house prices
sample_prices = np.random.normal(TRUE_MEAN, TRUE_STD, N_SAMPLE)

# Step 2: Compute basic summary statistics from the sample
x_bar = np.mean(sample_prices)               # sample mean
s     = np.std(sample_prices, ddof=1)        # sample std dev (ddof=1 for unbiased estimate)
n     = len(sample_prices)                   # sample size

print("Step 1-2: Sample statistics")
print(f"  Sample mean  (x̄): ${x_bar:,.2f}")
print(f"  Sample std   (s):  ${s:,.2f}")
print(f"  Sample size  (n):  {n}")
print()

# Step 3: Compute the Standard Error of the Mean
# SE tells us how much the sample mean is expected to vary across samples
SE = s / np.sqrt(n)

print("Step 3: Standard Error")
print(f"  SE = s / √n = ${s:,.2f} / √{n} = ${SE:,.2f}")
print(f"  Interpretation: sample means vary by ±${SE:,.0f} on average")
print()

# Step 4: Find the critical t-value for 95% CI, two-tailed
# degrees of freedom = n - 1 (we lose one df by estimating the mean)
CONFIDENCE = 0.95
ALPHA      = 1 - CONFIDENCE      # 0.05 — total probability in both tails
df         = n - 1               # degrees of freedom

# stats.t.ppf(0.975, df) gives the t-value with 2.5% in the upper tail
# (and 2.5% in the lower tail, for a two-tailed 95% interval)
t_critical = stats.t.ppf(1 - ALPHA/2, df=df)

print("Step 4: Critical t-value")
print(f"  Confidence level: {CONFIDENCE:.0%}")
print(f"  α = {ALPHA} → α/2 = {ALPHA/2} in each tail")
print(f"  Degrees of freedom: {df}")
print(f"  t* = stats.t.ppf(0.975, df={df}) = {t_critical:.4f}")
print(f"  (Compare: z* for large n would be 1.960)")
print()

# Step 5: Compute the Margin of Error
margin_of_error = t_critical * SE

print("Step 5: Margin of Error")
print(f"  MoE = t* × SE = {t_critical:.4f} × ${SE:,.2f} = ${margin_of_error:,.2f}")
print()

# Step 6: Compute the CI lower and upper bounds
ci_lower = x_bar - margin_of_error
ci_upper = x_bar + margin_of_error

print("Step 6: 95% Confidence Interval")
print(f"  CI = x̄ ± MoE = ${x_bar:,.2f} ± ${margin_of_error:,.2f}")
print(f"  CI = [${ci_lower:,.2f},  ${ci_upper:,.2f}]")
print()

# Step 7: Verify with scipy (should match our manual calculation)
ci_scipy = stats.t.interval(CONFIDENCE, df=df, loc=x_bar, scale=SE)

print("Step 7: Verification using scipy.stats.t.interval")
print(f"  scipy CI: [{ci_scipy[0]:,.2f},  {ci_scipy[1]:,.2f}]")
print(f"  Match: {'YES ✓' if abs(ci_lower - ci_scipy[0]) < 0.01 else 'NO ✗'}")
print()

# Does the CI capture the true mean?
captured = ci_lower <= TRUE_MEAN <= ci_upper
print(f"True mean (${TRUE_MEAN:,}) captured in CI: {'YES ✓' if captured else 'NO ✗'}")

## 5. The Key Demonstration: 100 CIs, ~5 Miss the Truth

This is the most powerful way to understand what a 95% confidence interval **actually means**.

We'll draw 100 independent samples of 50 houses, compute a 95% CI for each, and see how many of those 100 intervals actually contain the true mean (\$300,000).

The result will be approximately 95 successes and 5 misses — that's the definition of a 95% CI in action.


In [ ]:
# ============================================================
# DEMONSTRATION: Repeat 100 times — how many CIs capture the truth?
# ============================================================

np.random.seed(7)

N_REPETITIONS = 100    # number of times we repeat the sampling
N_EACH = 50           # houses sampled each time
CONF_LEVEL = 0.95     # 95% confidence level

# Storage for results
all_means  = []
all_lower  = []
all_upper  = []
captured   = []

for rep in range(N_REPETITIONS):
    # Draw a fresh sample of 50 houses
    sample = np.random.normal(TRUE_MEAN, TRUE_STD, N_EACH)

    # Compute sample statistics
    xbar = np.mean(sample)
    se   = np.std(sample, ddof=1) / np.sqrt(N_EACH)
    df   = N_EACH - 1

    # Compute 95% CI using scipy (efficient for 100 repetitions)
    lo, hi = stats.t.interval(CONF_LEVEL, df=df, loc=xbar, scale=se)

    # Record everything
    all_means.append(xbar)
    all_lower.append(lo)
    all_upper.append(hi)
    # Does this interval contain the true mean?
    captured.append(lo <= TRUE_MEAN <= hi)

# Count outcomes
n_captured = sum(captured)
n_missed   = N_REPETITIONS - n_captured

print("=" * 55)
print(f"100 REPETITIONS: Sample n={N_EACH}, CI={int(CONF_LEVEL*100)}%")
print("=" * 55)
print(f"CIs that CONTAIN the true mean (${TRUE_MEAN:,}): {n_captured}")
print(f"CIs that MISS the true mean:                   {n_missed}")
print(f"Coverage rate: {n_captured/N_REPETITIONS:.0%}")
print(f"Expected (by definition of 95% CI): ~95%")
print()
print("This empirically confirms what 95% confidence means:")
print("the PROCEDURE captures the truth ~95% of the time.")

In [ ]:
# ============================================================
# VISUALIZATION 1: The 100-CI plot — the classic CI diagram
# Green intervals contain the true mean; red ones miss it
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))

# Draw each CI as a horizontal line segment
for i in range(N_REPETITIONS):
    color  = '#27ae60' if captured[i] else '#e74c3c'  # green = hit, red = miss
    lw     = 1.5
    zorder = 3 if not captured[i] else 2  # draw red intervals on top

    # Draw the CI line
    ax.plot([all_lower[i], all_upper[i]], [i, i],
            color=color, linewidth=lw, zorder=zorder)

    # Draw a dot at the sample mean
    ax.plot(all_means[i], i, 'o', color=color, markersize=3, zorder=4)

# Draw the true mean as a vertical line
ax.axvline(TRUE_MEAN, color='navy', linewidth=2.5, linestyle='-',
           label=f'True mean = ${TRUE_MEAN:,}', zorder=5)

# Formatting
ax.set_xlabel('House Price ($)')
ax.set_ylabel('Sample Number (1 to 100)')
ax.set_title(f'100 Independent 95% CIs for Mean House Price (n={N_EACH} each)\n'
             f'{n_captured} intervals contain the true mean | {n_missed} miss it',
             fontsize=13)

# Format x-axis to show prices in $K
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Custom legend
green_patch = mpatches.Patch(color='#27ae60', label=f'Contains true mean ({n_captured} intervals)')
red_patch   = mpatches.Patch(color='#e74c3c', label=f'Misses true mean ({n_missed} intervals)')
true_line   = plt.Line2D([0], [0], color='navy', linewidth=2.5,
                          label=f'True mean = ${TRUE_MEAN:,}')
ax.legend(handles=[green_patch, red_patch, true_line], loc='lower right', fontsize=10)

# Add text box with the key interpretation lesson
textstr = ('95% CI Interpretation:\n'
           '"If we repeated this study many\n'
           'times, ~95% of the intervals we\n'
           'compute would contain the true mean."\n\n'
           'NOT: "There is a 95% probability\n'
           'the true mean is in THIS interval."')
ax.text(0.02, 0.98, textstr, transform=ax.transAxes,
        fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.show()

## 6. Width of the CI: The Effect of Sample Size

Recall the margin of error: $\text{MoE} = t^* \times \dfrac{s}{\sqrt{n}}$

Since $n$ is in the **denominator under a square root**, to halve the margin of error you need to **quadruple** the sample size.

| Sample size | Approximate MoE (at σ=\$50K, 95% CI) |
|------------|--------------------------------------|
| n = 10     | ±\$34,800 |
| n = 50     | ±\$13,900 |
| n = 200    | ±\$6,900  |
| n = 500    | ±\$4,400  |
| n = 2000   | ±\$2,200  |

This has a direct implication for ML: collecting more training data gives you **more precise estimates** of model performance and feature importances.


In [ ]:
# ============================================================
# DEMONSTRATION: CI width as a function of sample size
# We compare CIs built from n=50 vs n=200 vs n=500 samples
# ============================================================

np.random.seed(123)

# Sample sizes to compare
sample_sizes_to_compare = [50, 200, 500]
colors_n = ['#e74c3c', '#f39c12', '#27ae60']  # red, orange, green

# For each sample size, draw 50 repetitions and record CIs
results_by_n = {}

for n_size in sample_sizes_to_compare:
    reps_means  = []
    reps_lower  = []
    reps_upper  = []
    reps_width  = []
    reps_caught = []

    for _ in range(50):   # 50 repetitions per sample size
        sample = np.random.normal(TRUE_MEAN, TRUE_STD, n_size)
        xbar   = np.mean(sample)
        se     = np.std(sample, ddof=1) / np.sqrt(n_size)
        lo, hi = stats.t.interval(0.95, df=n_size-1, loc=xbar, scale=se)

        reps_means.append(xbar)
        reps_lower.append(lo)
        reps_upper.append(hi)
        reps_width.append(hi - lo)         # width = upper - lower
        reps_caught.append(lo <= TRUE_MEAN <= hi)

    results_by_n[n_size] = {
        'means' : np.array(reps_means),
        'lower' : np.array(reps_lower),
        'upper' : np.array(reps_upper),
        'widths': np.array(reps_width),
        'caught': reps_caught
    }

# Print summary statistics
print("=" * 60)
print("CI WIDTH COMPARISON (50 repetitions per sample size)")
print("=" * 60)
print(f"{'n':>6}  {'Mean CI Width':>16}  {'Median Width':>14}  {'Coverage':>10}")
print("-" * 60)

for n_size, data in results_by_n.items():
    mean_width   = np.mean(data['widths'])
    median_width = np.median(data['widths'])
    coverage     = sum(data['caught']) / 50
    print(f"{n_size:>6}  ${mean_width:>14,.0f}  ${median_width:>12,.0f}  {coverage:>10.0%}")

print()
print("Key relationship: Width ∝ 1/√n")
print(f"  n=50 → n=200: 4× more data → width shrinks by ~{1 - np.mean(results_by_n[200]['widths'])/np.mean(results_by_n[50]['widths']):.1%}")
print(f"  n=50 → n=500: 10× more data → width shrinks by ~{1 - np.mean(results_by_n[500]['widths'])/np.mean(results_by_n[50]['widths']):.1%}")

In [ ]:
# ============================================================
# VISUALIZATION 2: Side-by-side CI comparison for n=50, 200, 500
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(17, 8), sharey=True)

for ax, (n_size, data), color in zip(axes, results_by_n.items(), colors_n):
    for i in range(50):
        # Green if it captured the true mean, lighter colour if not
        c  = color if data['caught'][i] else '#bdc3c7'
        lw = 1.5
        ax.plot([data['lower'][i], data['upper'][i]], [i, i],
                color=c, linewidth=lw, alpha=0.85)
        ax.plot(data['means'][i], i, 'o', color=c, markersize=3)

    # True mean line
    ax.axvline(TRUE_MEAN, color='navy', linewidth=2, linestyle='-',
               label=f'True mean = ${TRUE_MEAN:,}')

    # Compute average CI width to display
    avg_width    = np.mean(data['widths'])
    n_captured_n = sum(data['caught'])

    ax.set_title(f'n = {n_size}\nAvg CI width: ${avg_width:,.0f}\n'
                 f'{n_captured_n}/50 capture true mean',
                 fontsize=11)
    ax.set_xlabel('Mean House Price')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.tick_params(axis='x', rotation=30)

    # Shade the range of all CIs to show overall spread
    left_extent  = np.min(data['lower'])
    right_extent = np.max(data['upper'])

axes[0].set_ylabel('Sample Number')

plt.suptitle('Effect of Sample Size on 95% CI Width\n'
             'Larger samples → narrower (more precise) intervals',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table for widths
print("Theoretical vs Observed CI Widths:")
for n_size in sample_sizes_to_compare:
    theoretical_width = 2 * stats.t.ppf(0.975, df=n_size-1) * (TRUE_STD / np.sqrt(n_size))
    observed_width    = np.mean(results_by_n[n_size]['widths'])
    print(f"  n={n_size}: theoretical=${theoretical_width:,.0f}, observed=${observed_width:,.0f}")

## 7. Effect Size: Cohen's d

### The Problem P-Values Don't Solve

Suppose you compare house prices in District A (near schools) vs District B (far from schools).

- **Test 1:** p = 0.001, difference = \$200. *Statistically significant, practically useless.*
- **Test 2:** p = 0.06, difference = \$40,000. *Not significant (barely), but potentially important!*

P-values cannot tell you which difference is more **meaningful**. Cohen's d can.

### What is Cohen's d?

Cohen's d is the **standardised difference** between two means — expressed in units of standard deviation:

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}$$

Where the pooled standard deviation is:

$$s_{\text{pooled}} = \sqrt{\frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}}$$

### Why Standardise?

- A \$10,000 difference means something very different for cheap apartments (std = \$15K) vs luxury homes (std = \$200K)
- Cohen's d accounts for the natural variability — it asks "how many standard deviations apart are the groups?"
- d = 1.0 means the groups differ by one full standard deviation — a very large difference

### Cohen's d Benchmarks (Cohen, 1988)

| d value | Interpretation | Visual |
|---------|---------------|--------|
| 0.0 – 0.2 | Negligible / very small | Distributions almost completely overlap |
| 0.2 – 0.5 | Small | Noticeable overlap, slight shift |
| 0.5 – 0.8 | Medium | Moderate separation |
| 0.8 – 1.2 | Large | Clear separation |
| > 1.2    | Very large | Distributions barely overlap |

**Note:** These benchmarks are rules of thumb. What counts as "meaningful" depends on the domain.


In [ ]:
# ============================================================
# COHEN'S d: Computing effect size for District A vs B
# ============================================================

np.random.seed(42)

# ---------------------------------------------------------------
# Helper function: compute Cohen's d from two samples
# ---------------------------------------------------------------
def cohens_d(group1, group2):
    """Compute Cohen's d (standardised mean difference) for two independent groups."""
    n1, n2   = len(group1), len(group2)
    mean1    = np.mean(group1)
    mean2    = np.mean(group2)
    var1     = np.var(group1, ddof=1)   # sample variance (unbiased)
    var2     = np.var(group2, ddof=1)

    # Pooled standard deviation — weighted average of both variances
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    # Standardised difference
    d = (mean1 - mean2) / pooled_std
    return d, pooled_std

# ---------------------------------------------------------------
# Helper function: classify effect size
# ---------------------------------------------------------------
def classify_effect(d_abs):
    """Return a string label for the magnitude of Cohen's d."""
    if d_abs < 0.2:    return 'negligible'
    elif d_abs < 0.5:  return 'small'
    elif d_abs < 0.8:  return 'medium'
    elif d_abs < 1.2:  return 'large'
    else:              return 'very large'

# ---------------------------------------------------------------
# Scenario 1: Large sample, tiny effect — statistically "significant"
# ---------------------------------------------------------------
N_LARGE = 5000   # 5000 houses — very large dataset
district_A_large = np.random.normal(300_000, 50_000, N_LARGE)
district_B_large = np.random.normal(300_500, 50_000, N_LARGE)  # Only $500 difference!

t1, p1 = stats.ttest_ind(district_A_large, district_B_large)
d1, pooled_std1 = cohens_d(district_A_large, district_B_large)

# ---------------------------------------------------------------
# Scenario 2: Moderate sample, large effect — borderline "significant"
# ---------------------------------------------------------------
N_MOD = 40   # only 40 houses — small survey
district_A_mod = np.random.normal(300_000, 50_000, N_MOD)
district_B_mod = np.random.normal(345_000, 50_000, N_MOD)  # $45,000 difference!

t2, p2 = stats.ttest_ind(district_A_mod, district_B_mod)
d2, pooled_std2 = cohens_d(district_A_mod, district_B_mod)

# ---------------------------------------------------------------
# Display comparison
# ---------------------------------------------------------------
print("=" * 70)
print("COHEN'S d vs P-VALUE: Same p ≠ Same Practical Importance")
print("=" * 70)
print()
print(f"SCENARIO 1: Huge dataset (n={N_LARGE} per group), tiny difference")
print(f"  True difference : $500 (0.17% of house price)")
print(f"  Observed diff   : ${abs(np.mean(district_A_large) - np.mean(district_B_large)):,.0f}")
print(f"  p-value         : {p1:.6f}  ← {'SIGNIFICANT' if p1 < 0.05 else 'not significant'}")
print(f"  Cohen's d       : {abs(d1):.4f}  ← {classify_effect(abs(d1)).upper()} effect")
print(f"  Practical use?  : NO — a $500 difference in a $300K house is irrelevant")
print()
print(f"SCENARIO 2: Small dataset (n={N_MOD} per group), large difference")
print(f"  True difference : $45,000 (15% of house price)")
print(f"  Observed diff   : ${abs(np.mean(district_A_mod) - np.mean(district_B_mod)):,.0f}")
print(f"  p-value         : {p2:.4f}  ← {'SIGNIFICANT' if p2 < 0.05 else 'borderline / not significant'}")
print(f"  Cohen's d       : {abs(d2):.4f}  ← {classify_effect(abs(d2)).upper()} effect")
print(f"  Practical use?  : YES — $45K matters enormously for house purchases")
print()
print("LESSON: Low p doesn't mean large effect; high p doesn't mean small effect.")
print("Always report BOTH p-value AND Cohen's d (or equivalent effect size).")

In [ ]:
# ============================================================
# VISUALIZATION 3: What Different Cohen's d Values Look Like
# Show overlapping distributions for various d values
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()  # flatten to 1D for easy iteration

# Effect sizes to visualise
d_values = [0.0, 0.2, 0.5, 0.8, 1.2, 2.0]
d_labels = ['d=0.0\n(Negligible)', 'd=0.2\n(Small)',
             'd=0.5\n(Medium)',    'd=0.8\n(Large)',
             'd=1.2\n(Very Large)', 'd=2.0\n(Extreme)']

# Use a standard normal for District A; shift District B by d standard deviations
x = np.linspace(-5, 7, 1000)   # x-axis: standardised units

for ax, d, label in zip(axes, d_values, d_labels):
    # District A: standard normal N(0, 1)
    pdf_A = stats.norm.pdf(x, loc=0, scale=1)
    # District B: shifted by d standard deviations
    pdf_B = stats.norm.pdf(x, loc=d, scale=1)

    # Plot both distributions
    ax.plot(x, pdf_A, color='steelblue', linewidth=2, label='District A')
    ax.plot(x, pdf_B, color='tomato',    linewidth=2, label='District B')

    # Fill the overlap area to show how much they share
    overlap_y = np.minimum(pdf_A, pdf_B)
    ax.fill_between(x, overlap_y, alpha=0.35, color='purple', label='Overlap')

    # Mark the means with vertical dashed lines
    ax.axvline(0, color='steelblue', linestyle='--', alpha=0.6)
    ax.axvline(d, color='tomato',    linestyle='--', alpha=0.6)

    # Compute the percentage of area that overlaps (overlap coefficient)
    # Approximate numerically
    dx_step = x[1] - x[0]
    overlap_coeff = np.sum(overlap_y) * dx_step  # area under overlap curve

    ax.set_title(f'{label}\nOverlap ≈ {overlap_coeff:.0%}', fontsize=11)
    ax.set_xlabel('Standardised house price (σ units)')
    ax.set_ylabel('Probability density')
    ax.set_ylim(0, 0.45)
    ax.set_xlim(-4, 6)

    if d == 0.0:  # only show legend on first panel
        ax.legend(fontsize=9)

plt.suptitle('Cohen\'s d Effect Sizes: What They Look Like as Overlapping Distributions\n'
             'District A (blue) vs District B (red)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. The "Significant but Trivial" Paradox

This paradox is especially important in modern big-data ML. When you have millions of rows:

- Almost **every feature** will be statistically significant (even random noise with p ≈ 0.001)
- Most of those features will have **negligible Cohen's d**
- Including them in your model may hurt generalisation (overfitting) without improving predictions

### ML Implication

When comparing models or features on a large dataset:
1. Compute p-value to see if the difference is real (non-zero)
2. Compute Cohen's d to see if the difference is **big enough to matter**
3. Report the 95% CI on the improvement — this shows the **range of plausible improvement**

If Model A beats Model B by 0.01% AUC (p = 0.0001, d = 0.01), that's statistically real but practically meaningless.


In [ ]:
# ============================================================
# VISUALIZATION 4: The "Significant but Trivial" Paradox
# Show two scenarios side by side:
#   Left: LARGE n, tiny difference, very small p, tiny d
#   Right: SMALL n, large difference, larger p, large d
# ============================================================

np.random.seed(88)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ---------------------------------------------------------------
# LEFT: Big data, tiny effect
# ---------------------------------------------------------------
ax = axes[0]

# Population parameters: $300 difference in a $300K market (negligible)
n_big = 10_000
mu_A_big, mu_B_big = 300_000, 300_300
sigma_big = 50_000

sample_A_big = np.random.normal(mu_A_big, sigma_big, n_big)
sample_B_big = np.random.normal(mu_B_big, sigma_big, n_big)

t_big, p_big = stats.ttest_ind(sample_A_big, sample_B_big)
d_big, _     = cohens_d(sample_A_big, sample_B_big)

# Plot histograms of the two groups
bins_big = np.linspace(100_000, 500_000, 80)
ax.hist(sample_A_big, bins=bins_big, alpha=0.5, color='steelblue',
        density=True, label=f'District A: mean=${np.mean(sample_A_big):,.0f}')
ax.hist(sample_B_big, bins=bins_big, alpha=0.5, color='tomato',
        density=True, label=f'District B: mean=${np.mean(sample_B_big):,.0f}')

# Mark the means
ax.axvline(np.mean(sample_A_big), color='steelblue', linewidth=2, linestyle='--')
ax.axvline(np.mean(sample_B_big), color='tomato',    linewidth=2, linestyle='--')

ax.set_title(f'SCENARIO 1: Big Data (n={n_big:,} per group)\n'
             f'True difference: ${mu_B_big - mu_A_big:,}\n'
             f'p = {p_big:.2e}  ← highly significant!\n'
             f'Cohen\'s d = {abs(d_big):.4f}  ← {classify_effect(abs(d_big))} effect',
             fontsize=10)
ax.set_xlabel('House Price ($)')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(fontsize=9)

# Add verdict box
ax.text(0.04, 0.97,
        f'VERDICT:\np={p_big:.2e} says SIGNIFICANT\nd={abs(d_big):.4f} says NEGLIGIBLE\n'
        'A $300 difference in a\n$300K market is irrelevant!',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#fdedec', alpha=0.9))

# ---------------------------------------------------------------
# RIGHT: Small data, large effect
# ---------------------------------------------------------------
ax = axes[1]

# Population parameters: $50,000 difference — a big real effect
n_small = 30
mu_A_small, mu_B_small = 300_000, 350_000
sigma_small = 50_000

sample_A_small = np.random.normal(mu_A_small, sigma_small, n_small)
sample_B_small = np.random.normal(mu_B_small, sigma_small, n_small)

t_small, p_small = stats.ttest_ind(sample_A_small, sample_B_small)
d_small, _       = cohens_d(sample_A_small, sample_B_small)

# Plot as smooth KDE for small n
x_range = np.linspace(100_000, 550_000, 400)
kde_A    = stats.gaussian_kde(sample_A_small)
kde_B    = stats.gaussian_kde(sample_B_small)

ax.plot(x_range, kde_A(x_range), color='steelblue', linewidth=2.5,
        label=f'District A: mean=${np.mean(sample_A_small):,.0f}')
ax.plot(x_range, kde_B(x_range), color='tomato',    linewidth=2.5,
        label=f'District B: mean=${np.mean(sample_B_small):,.0f}')

# Fill under curves
ax.fill_between(x_range, kde_A(x_range), alpha=0.25, color='steelblue')
ax.fill_between(x_range, kde_B(x_range), alpha=0.25, color='tomato')

# Mark the means
ax.axvline(np.mean(sample_A_small), color='steelblue', linewidth=2, linestyle='--')
ax.axvline(np.mean(sample_B_small), color='tomato',    linewidth=2, linestyle='--')

ax.set_title(f'SCENARIO 2: Small Data (n={n_small} per group)\n'
             f'True difference: ${mu_B_small - mu_A_small:,}\n'
             f'p = {p_small:.4f}  ← {"significant" if p_small < 0.05 else "borderline / not significant"}\n'
             f'Cohen\'s d = {abs(d_small):.4f}  ← {classify_effect(abs(d_small))} effect',
             fontsize=10)
ax.set_xlabel('House Price ($)')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(fontsize=9)

# Add verdict box
ax.text(0.04, 0.97,
        f'VERDICT:\np={p_small:.4f} says {"SIGNIFICANT" if p_small < 0.05 else "borderline"}\n'
        f'd={abs(d_small):.3f} says {classify_effect(abs(d_small)).upper()}\n'
        'A $50K difference in a\n$300K market MATTERS!',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#d5f5e3', alpha=0.9))

plt.suptitle('The "Significant but Trivial" Paradox\n'
             'P-value alone is insufficient — always report Cohen\'s d',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Confidence Intervals as a Visual Significance Test

Another way to use CIs: **compare two groups by looking at whether their CIs overlap**.

### The Rules

| CI overlap | Informal conclusion |
|-----------|---------------------|
| CIs don't overlap at all | Strong evidence of a significant difference |
| CIs overlap slightly | Borderline — formally test to be sure |
| CIs overlap substantially | Likely no significant difference |

**Caution:** This is an approximation. Two 95% CIs can overlap even when a formal test finds p < 0.05 (especially for comparing means). For a rigorous test, always run the formal test AND report the CI on the **difference**, not on each group separately.

The best practice: report the 95% CI on the **difference between means**, not just each mean's CI.


In [ ]:
# ============================================================
# VISUALIZATION 5: CI Overlap as a Visual Significance Test
# Compare three scenarios: no overlap, slight overlap, large overlap
# ============================================================

np.random.seed(21)

fig, axes = plt.subplots(1, 3, figsize=(17, 7))

# Define three scenarios with different district price separations
scenarios = [
    {'label': 'Strong Evidence\n(No CI Overlap)',       'mean_A': 300_000, 'mean_B': 370_000, 'n': 100},
    {'label': 'Borderline\n(Slight CI Overlap)',         'mean_A': 300_000, 'mean_B': 330_000, 'n': 100},
    {'label': 'No Evidence\n(Substantial CI Overlap)',   'mean_A': 300_000, 'mean_B': 310_000, 'n': 100},
]

SIGMA = 50_000   # same standard deviation across all scenarios

for ax, scenario in zip(axes, scenarios):
    n = scenario['n']

    # Generate samples
    grp_A = np.random.normal(scenario['mean_A'], SIGMA, n)
    grp_B = np.random.normal(scenario['mean_B'], SIGMA, n)

    # Compute CIs for each group
    def compute_ci(data, conf=0.95):
        """Return (mean, lower_ci, upper_ci) for a sample."""
        m  = np.mean(data)
        se = np.std(data, ddof=1) / np.sqrt(len(data))
        lo, hi = stats.t.interval(conf, df=len(data)-1, loc=m, scale=se)
        return m, lo, hi

    mean_A, lo_A, hi_A = compute_ci(grp_A)
    mean_B, lo_B, hi_B = compute_ci(grp_B)

    # Compute CI on the DIFFERENCE (the right way to test)
    diff_mean = mean_B - mean_A
    se_diff   = np.sqrt((np.var(grp_A, ddof=1)/n) + (np.var(grp_B, ddof=1)/n))
    df_diff   = 2*n - 2
    lo_diff, hi_diff = stats.t.interval(0.95, df=df_diff, loc=diff_mean, scale=se_diff)

    # Formal p-value
    _, p_formal = stats.ttest_ind(grp_A, grp_B)

    # Plot the CI for each group as error bars
    ax.errorbar(['District A', 'District B'],
                [mean_A, mean_B],
                yerr=[[mean_A - lo_A, mean_B - lo_B],
                      [hi_A - mean_A, hi_B - mean_B]],
                fmt='o', capsize=10, capthick=3, linewidth=2.5,
                markersize=10,
                color=['steelblue', 'tomato'][0])  # one colour for clarity

    # Colour code the points: blue for A, red for B
    ax.plot('District A', mean_A, 'o', color='steelblue', markersize=12, zorder=5)
    ax.plot('District B', mean_B, 'o', color='tomato',    markersize=12, zorder=5)

    # Shade the CI bars separately for clarity
    ax.vlines('District A', lo_A, hi_A, color='steelblue', linewidth=4, alpha=0.5)
    ax.vlines('District B', lo_B, hi_B, color='tomato',    linewidth=4, alpha=0.5)

    # Add horizontal lines at the CI bounds to make overlap obvious
    ax.axhline(hi_A, color='steelblue', linestyle=':', alpha=0.5)
    ax.axhline(lo_B, color='tomato',    linestyle=':', alpha=0.5)

    # Determine overlap colour for title
    overlap_exists = hi_A > lo_B
    title_color = '#27ae60' if not overlap_exists else ('#e67e22' if overlap_exists and p_formal < 0.05 else '#e74c3c')

    # Format title with stats
    ax.set_title(f'{scenario["label"]}\n'
                 f'True diff: ${scenario["mean_B"] - scenario["mean_A"]:,}\n'
                 f'Observed diff: ${diff_mean:,.0f}  p={p_formal:.4f}\n'
                 f'95% CI on difference: [${lo_diff:,.0f}, ${hi_diff:,.0f}]',
                 fontsize=9)

    ax.set_ylabel('Mean House Price')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

    # Annotate whether CIs overlap
    if not overlap_exists:
        ax.text(0.5, 0.06, '✓ No overlap → Significant',
                transform=ax.transAxes, ha='center', fontsize=10,
                color='#27ae60', fontweight='bold')
    else:
        ax.text(0.5, 0.06, '↕ Overlap present',
                transform=ax.transAxes, ha='center', fontsize=10,
                color='#e74c3c', fontweight='bold')

plt.suptitle('95% CIs as Visual Test: Overlap of Group CIs\n'
             'Error bars show 95% CI for each group mean (n=100 per group)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Best practice: Report CI on the DIFFERENCE (last line of each scenario title),")
print("not just CIs on each group mean. The CI on the difference is the definitive answer.")

In [ ]:
# ============================================================
# FINAL SUMMARY: Complete example combining all concepts
# A data scientist's full report: CI + p-value + Cohen's d
# ============================================================

np.random.seed(42)

print("=" * 65)
print("FULL ANALYSIS EXAMPLE: House Prices in District A vs District B")
print("=" * 65)
print()

# Generate realistic house price samples
n_A = 120                          # 120 houses in District A
n_B = 95                           # 95 houses in District B

sample_A_final = np.random.normal(305_000, 48_000, n_A)   # near-school district
sample_B_final = np.random.normal(285_000, 55_000, n_B)   # far-from-school district

# --- Compute all statistics ---

# Means
mean_A = np.mean(sample_A_final)
mean_B = np.mean(sample_B_final)
diff   = mean_A - mean_B

# 95% CI for each group's mean
se_A  = np.std(sample_A_final, ddof=1) / np.sqrt(n_A)
se_B  = np.std(sample_B_final, ddof=1) / np.sqrt(n_B)
ci_A  = stats.t.interval(0.95, df=n_A-1, loc=mean_A, scale=se_A)
ci_B  = stats.t.interval(0.95, df=n_B-1, loc=mean_B, scale=se_B)

# 95% CI for the difference
se_diff_final = np.sqrt(se_A**2 + se_B**2)
df_final      = n_A + n_B - 2
t_crit_final  = stats.t.ppf(0.975, df=df_final)
ci_diff_lo    = diff - t_crit_final * se_diff_final
ci_diff_hi    = diff + t_crit_final * se_diff_final

# T-test
t_final, p_final = stats.ttest_ind(sample_A_final, sample_B_final)

# Cohen's d
d_final, pooled = cohens_d(sample_A_final, sample_B_final)

# --- Print the complete report ---
print("DESCRIPTIVE STATISTICS")
print(f"  District A: n={n_A}, mean=${mean_A:,.0f}, std=${np.std(sample_A_final, ddof=1):,.0f}")
print(f"  District B: n={n_B}, mean=${mean_B:,.0f}, std=${np.std(sample_B_final, ddof=1):,.0f}")
print()
print("CONFIDENCE INTERVALS (95%)")
print(f"  District A mean: [${ci_A[0]:,.0f},  ${ci_A[1]:,.0f}]")
print(f"  District B mean: [${ci_B[0]:,.0f},  ${ci_B[1]:,.0f}]")
print(f"  Difference (A-B): [${ci_diff_lo:,.0f},  ${ci_diff_hi:,.0f}]")
print()
print("HYPOTHESIS TEST")
print(f"  H₀: True mean prices are equal")
print(f"  H₁: True mean prices differ")
print(f"  t-statistic: {t_final:.3f}")
print(f"  p-value: {p_final:.4f}")
print(f"  Decision: {'Reject H₀' if p_final < 0.05 else 'Fail to reject H₀'} at α=0.05")
print()
print("EFFECT SIZE")
print(f"  Mean difference: ${diff:,.0f}")
print(f"  Pooled std:      ${pooled:,.0f}")
print(f"  Cohen's d:       {abs(d_final):.3f}  ({classify_effect(abs(d_final))} effect)")
print()
print("INTERPRETATION")
print(f"  Houses in District A cost on average ${diff:,.0f} more than District B.")
print(f"  This difference is {'statistically significant (p < 0.05)' if p_final < 0.05 else 'not statistically significant (p ≥ 0.05)'}.")
print(f"  The effect size is {classify_effect(abs(d_final))} (d={abs(d_final):.3f}).")
print(f"  With 95% confidence, the true price advantage of District A is")
print(f"  between ${ci_diff_lo:,.0f} and ${ci_diff_hi:,.0f}.")
print()
print("This is the correct, complete way to report statistical findings.")
print("Never report just the p-value alone.")

## 10. Self-Check Questions

---

### Question 1
A 95% CI for mean house price is [\$295K, \$325K]. Your analyst says: *"There's a 95% probability the true mean is in this range."* Is this correct?

<details>
<summary>Click to reveal answer</summary>

**No — this is the most common CI misinterpretation.**

Once you've computed the interval [\$295K, \$325K], the true mean either IS in that interval or it ISN'T. There is no probability to assign to a fixed interval — it's a binary fact.

**The correct interpretation:**
> *"We used a procedure that, if repeated across many different samples, would produce intervals containing the true mean approximately 95% of the time. This specific interval [\$295K, \$325K] is one result of that procedure."*

The 95% refers to the **long-run reliability of the procedure**, not the probability that the true mean is inside this particular interval.

Practical alternative (easier to communicate): *"We are 95% confident in the procedure that generated this interval, and we can say the data are consistent with a true mean anywhere between \$295K and \$325K."*
</details>

---

### Question 2
You compare two ML models. Model A beats Model B with p = 0.02 and Cohen's d = 0.05. Should you switch to Model A? Why or why not?

<details>
<summary>Click to reveal answer</summary>

**Probably not.** Here's why:

- p = 0.02: The difference is statistically real — it's unlikely to be sampling noise
- d = 0.05: The effect is **negligible** — the distributions of performance metrics overlap almost completely

Cohen's d = 0.05 means Model A's average performance is only 0.05 standard deviations better. In practice:
- The improvement may not hold on new data
- Model A might be more complex, less interpretable, or slower
- The 95% CI on the improvement would be very narrow around a tiny number

**Decision framework:**
1. Is the improvement statistically real? → Yes (p = 0.02)
2. Is the improvement practically meaningful? → No (d = 0.05)
3. What's the confidence interval on the improvement? → Probably [0.1%, 0.3%] or similar tiny range
4. Does Model A have other costs? → Likely
5. **Conclusion:** Keep Model B. The improvement doesn't justify the complexity.
</details>

---

### Question 3
How would you make your 95% confidence interval narrower without changing the confidence level?

<details>
<summary>Click to reveal answer</summary>

Recall: $\text{MoE} = t^* \times \dfrac{s}{\sqrt{n}}$

To narrow the CI (reduce MoE) without changing confidence level (which fixes $t^*$), you have two options:

**1. Increase sample size (n) ↑**
- Larger $n$ → larger $\sqrt{n}$ → smaller MoE
- To halve the interval, quadruple $n$
- This is the most direct, controllable approach in practice

**2. Reduce variability (s) ↓**
- If the population itself is less variable, $s$ decreases
- In study design: focus on a more homogeneous population (e.g., test only 3-bedroom houses rather than all houses)
- Use better measurement instruments to reduce measurement error
- Use matched-pairs or repeated-measures designs (controls for between-subject variability)

**Note:** Changing the confidence level (e.g., from 95% to 90%) WOULD narrow the interval, but that changes the question being asked. At 90% CI, you accept a higher error rate (10% of CIs miss the true value).
</details>

---

### Question 4
You have two districts with 95% CIs for mean price:  
- District A: [\$295K, \$315K]  
- District B: [\$305K, \$330K]  

The CIs overlap between \$305K and \$315K. Can you conclude there's no significant difference?

<details>
<summary>Click to reveal answer</summary>

**No — overlapping CIs do NOT prove non-significance.**

Two 95% CIs can overlap even when the formal two-sample t-test finds p < 0.05. This happens because:
- Each CI is built for its own mean, not for the difference
- The CI on the **difference** is the correct tool for this question

In this case:
- District A mean ≈ \$305K
- District B mean ≈ \$317.5K
- The means differ by ≈ \$12.5K
- Whether this is significant depends on the formal test

**Rule of thumb (approximate):**
- If 95% CIs have **zero overlap**: definitely significant at p < 0.05
- If 95% CIs **slightly overlap**: may or may not be significant — run the test
- If 95% CIs **heavily overlap**: unlikely to be significant

**Best practice:** Always report the 95% CI on the **difference** [\$lo_diff, \$hi_diff]. If this CI excludes 0, the difference is significant.
</details>


## Summary

| Concept | Formula | Key Takeaway |
|---------|---------|-------------|
| **Standard Error** | $SE = s/\sqrt{n}$ | How much sample means vary |
| **95% CI** | $\bar{x} \pm t^* \times SE$ | Range of plausible true values |
| **CI interpretation** | Long-run coverage = 95% | About the PROCEDURE, not this interval |
| **CI width** | $\propto 1/\sqrt{n}$ | 4× data → 2× precision |
| **Cohen's d** | $(\bar{x}_1 - \bar{x}_2) / s_{pooled}$ | Standardised effect size |
| **d = 0.2** | Small effect | Barely visible difference |
| **d = 0.5** | Medium effect | Noticeable difference |
| **d = 0.8** | Large effect | Clear separation |
| **Complete report** | p + CI + d | Never report p-value alone |

---

### The Complete Reporting Template

```
"[Group A] was [higher/lower] than [Group B] by [raw difference],
 95% CI [lower, upper],  t(df) = [t], p = [p], d = [d] ([small/medium/large] effect)."
```

Example:  
*"District A house prices were higher than District B by \$20,000,  
95% CI [\$12,000, \$28,000], t(213) = 4.87, p = 0.0001, d = 0.42 (small-medium effect)."*

This single sentence tells you: the direction, the magnitude, the precision, the evidence strength, and the practical importance. It is a complete, honest, interpretable statistical summary.

---

**Week 3 complete!** You now understand the most important statistical concepts for responsible ML:
- Hypothesis testing and p-values (Notebook 5)
- Confidence intervals and effect sizes (this notebook)
